# Feature engineering

Ce notebook transforme les données nettoyées d’incendies et de communes en variables spatio-temporelles prêtes pour la modélisation.

- **Workflow :** étape entre le nettoyage des données et la modélisation.
- **Input :** `incendies.parquet` et `communes_metropole_corse.parquet`.
- **Output :** `incendies_features_v2.parquet`, agrégé au niveau commune-mois.

Etapes :
- fusion des incendies et des communes ;
- création d’une grille commune × année × mois ;
- création des variables cibles ;
- variables temporelles cycliques ;
- variables historiques décalées ;
- scores de risque causaux ;
- indice de contagion spatiale à 30 km ;
- variables calendaires ;
- optimisation mémoire ;
- xport du dataset final.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

# adds parent file of the current directory
# to the paths in which Python looks for modules to import
# in the current Python process
sys.path.append(str(Path.cwd().parent))

from src.config import (
    BDIFF_CLEAN_DIR,
    DATA_DIR,
    GEO_DATA_CLEAN_DIR,
    SPATIO_TEMP_DATA_DIR,
)
from utils.dataset_utils import export_versioned_dataset
from utils.feature_utils import add_monthly_calendar_features


In [15]:
df_incendies = pd.read_parquet(
    BDIFF_CLEAN_DIR / "incendies.parquet"
)

df_communes = pd.read_parquet(
    GEO_DATA_CLEAN_DIR / "communes_metropole_corse.parquet"
)

#### Fusion des dataset
- incendies/BDIFF
- liste des communes/data.gouv.fr
- sur la clé code_insee

In [16]:
df = df_incendies.merge(
    df_communes,
    on="code_insee",
    how="left"
)

In [17]:
df['annee'] = df['date_de_premiere_alerte'].dt.year
df['mois'] = df['date_de_premiere_alerte'].dt.month
df['jours'] = df['date_de_premiere_alerte'].dt.day

#conversion de la surface en hectares (1ha = 10000 m2)
df['surface_parcourue_ha'] = df['surface_parcourue_m2'] / 10_000

### construction de la grille spatio-temporelle au niveau communal (**code\_insee**)

#### Objectif

- Générer une grille complète Commune $\times$ Année $\times$ Mois<br>
pour inclure toutes les communes, y compris celles qui n'ont jamais connu d'incendie (exemples négatifs $Y = 0$)<br>


- Définir les variables cibles (Targets) :
    - `target_occurrence` : Variable binaire ($1$ si au moins un feu dans le mois, $0$ sinon)
    - `nb_incendies` : Comptage des départs de feu
    - `surface_brulee_ha` : Surface totale brûlée (et sa version `log1p`)

- Créer les variables explicatives (Features) :
    - Temporelles : Variables cycliques (sin_mois, cos_mois), indicateurs de saisonnalité et de vacances
    - Spatiales et socio-démographiques : Coordonnées, densité, altitude, population
    - Historiques de risque : Calculées uniquement sur le passé (jusqu'à $t-1$) pour éviter tout Data Leakage.

In [18]:
# -----------------------------------------------------------------------------
# 1. PRÉPARATION DES RÉFÉRENTIELS ET DE LA GRILLE SPATIO-TEMPORELLE
# -----------------------------------------------------------------------------

# Extraction des communes uniques à partir de vos données nettoyées
df_communes = df[[
    'code_insee', 'latitude', 'longitude', 'population',
    'superficie_hectare', 'densite', 'altitude_moyenne'
]].drop_duplicates(subset=['code_insee']).copy()

all_insee = df_communes['code_insee'].unique()
years = sorted([int(y) for y in df['annee'].dropna().unique()])
months = list(range(1, 13))

# Grille complète : Produit cartésien Commune x Année x Mois
grid_index = pd.MultiIndex.from_product(
    [all_insee, years, months],
    names=['code_insee', 'annee', 'mois']
)
df_grid = pd.DataFrame(index=grid_index).reset_index()

# -----------------------------------------------------------------------------
# 2. AGRÉGATION DES INCENDIES (BDIFF) SUR LA GRILLE
# -----------------------------------------------------------------------------

df_bdiff_agg = (
    df.groupby(['code_insee', 'annee', 'mois'])
    .agg(
        nb_incendies=('surface_parcourue_ha', 'count'),
        surface_brulee_ha=('surface_parcourue_ha', 'sum')
    )
    .reset_index()
)

# Jointure de la grille avec l'historique d'incendies
df_spatio_temp = pd.merge(
    df_grid,
    df_bdiff_agg,
    on=['code_insee', 'annee', 'mois'],
    how='left'
)

# Remplacement des valeurs manquantes (absence d'incendie = 0)
df_spatio_temp['nb_incendies'] = df_spatio_temp['nb_incendies'].fillna(0).astype(int)
df_spatio_temp['surface_brulee_ha'] = df_spatio_temp['surface_brulee_ha'].fillna(0)

# Target binaire (Occurrence d'incendie)
df_spatio_temp['target_occurrence'] = (df_spatio_temp['nb_incendies'] > 0).astype(int)
# Target log-transformée pour les modèles de régression
df_spatio_temp['target_log_surface'] = np.log1p(df_spatio_temp['surface_brulee_ha'])

# -----------------------------------------------------------------------------
# 3. ENRICHISSEMENT AVEC LES FEATURES TEMPORELLES ET CYCLIQUES
# -----------------------------------------------------------------------------

# Transformations sin/cos pour capturer la continuité des mois
df_spatio_temp['sin_mois'] = np.sin(2 * np.pi * df_spatio_temp['mois'] / 12)
df_spatio_temp['cos_mois'] = np.cos(2 * np.pi * df_spatio_temp['mois'] / 12)

# Indicateurs de saison forte et de vacances
df_spatio_temp['is_summer_peak'] = df_spatio_temp['mois'].isin([7, 8]).astype(int)
df_spatio_temp['is_spring_peak'] = df_spatio_temp['mois'].isin([3, 4]).astype(int)

# -----------------------------------------------------------------------------
# 4. ENRICHISSEMENT AVEC LES DONNÉES STATIQUES DE LA COMMUNE
# -----------------------------------------------------------------------------

df_spatio_temp = pd.merge(df_spatio_temp, df_communes, on='code_insee', how='left')

# -----------------------------------------------------------------------------
# 5. FEATURES HISTORIQUES GLISSANTES (SANS DATA LEAKAGE)
# -----------------------------------------------------------------------------

# Tri temporel strict
df_spatio_temp = df_spatio_temp.sort_values(by=['code_insee', 'annee', 'mois']).reset_index(drop=True)

# Nombre cumulé d'incendies dans la commune lors des 12 derniers mois (décalé de 1 mois)
df_spatio_temp['feux_commune_last_12m'] = (
    df_spatio_temp.groupby('code_insee')['nb_incendies']
    .transform(lambda x: x.shift(1).rolling(12, min_periods=1).sum())
    .fillna(0)
)

# Surface cumulée brûlée dans la commune lors des 12 derniers mois (décalée de 1 mois)
df_spatio_temp['surface_commune_last_12m'] = (
    df_spatio_temp.groupby('code_insee')['surface_brulee_ha']
    .transform(lambda x: x.shift(1).rolling(12, min_periods=1).sum())
    .fillna(0)
)


In [19]:
# Lags temporels récents communaux (M-1, M-2, M-3)
df_spatio_temp = df_spatio_temp.sort_values(['code_insee', 'annee', 'mois'])

df_spatio_temp['feux_commune_last_1m'] = df_spatio_temp.groupby('code_insee')['target_occurrence'].shift(1).fillna(0).astype('int8')
df_spatio_temp['feux_commune_last_2m'] = df_spatio_temp.groupby('code_insee')['target_occurrence'].shift(2).fillna(0).astype('int8')
df_spatio_temp['feux_commune_last_3m'] = df_spatio_temp.groupby('code_insee')['target_occurrence'].shift(3).fillna(0).astype('int8')

### Score de risque historique intrinsèque

Les scores sont calculés de manière causale sur l'historique disponible avant chaque mois.

In [20]:
# Scores de risque causaux : chaque ligne ne voit que les mois précédents.
df_spatio_temp = df_spatio_temp.sort_values(['code_insee', 'annee', 'mois']).reset_index(drop=True)

prior_target = df_spatio_temp.groupby('code_insee')['target_occurrence'].shift(1)
df_spatio_temp['score_risque_commune_global'] = (
    prior_target.groupby(df_spatio_temp['code_insee']).transform(
        lambda values: values.expanding(min_periods=1).mean()
    ).fillna(0).astype('float32')
)

prior_month_target = df_spatio_temp.groupby(['code_insee', 'mois'])['target_occurrence'].shift(1)
df_spatio_temp['score_risque_commune_mensuel'] = (
    prior_month_target.groupby(
        [df_spatio_temp['code_insee'], df_spatio_temp['mois']]
    ).transform(
        lambda values: values.expanding(min_periods=1).mean()
    ).fillna(0).astype('float32')
)

print("Scores de risque causaux calculés sans utiliser la cible du mois courant.")

 Score de risque historique calculé sans fuite de données (période 2011–2021).


### Indice de Contagion Spatiale par Distance (Feux voisins à $M-1$)
utilise un arbre de recherche spatiale (cKDTree) sur les coordonnées GPS des communes<br>
pour mesurer l'activité d'incendie dans un rayon de 30 km au mois précédent ($M-1$)

In [ ]:
# Indice de contagion spatiale (Propagation du risque à M-1)

# 1. Tri chronologique et création de l'index temporel mensuel
df_spatio_temp = df_spatio_temp.sort_values(['annee', 'mois', 'code_insee'])
df_spatio_temp['date_idx'] = df_spatio_temp['annee'] * 12 + df_spatio_temp['mois']

# 2. Création du tableau pivoté (Lignes = Mois, Colonnes = Communes)
pivot_df = df_spatio_temp.pivot(index='date_idx', columns='code_insee', values='target_occurrence').fillna(0)
commune_codes = pivot_df.columns  # Ordre exact des communes dans les colonnes de la matrice
pivoted_fires = pivot_df.values
n_dates, n_communes = pivoted_fires.shape

# 3. Alignement parfait de coords_df sur l'ordre exact de commune_codes
coords_df = df_communes[['code_insee', 'latitude', 'longitude']].drop_duplicates('code_insee')
coords_df = coords_df.set_index('code_insee').reindex(commune_codes).reset_index()

# Remplissage par sécurité des coordonnées manquantes par la moyenne globale
coords_df['latitude'] = coords_df['latitude'].fillna(coords_df['latitude'].mean())
coords_df['longitude'] = coords_df['longitude'].fillna(coords_df['longitude'].mean())

# 4. Conversion degrés -&gt; kilomètres
lat_mean = coords_df['latitude'].mean()
coords_km = np.column_stack([
    coords_df['longitude'].values * 111.0 * np.cos(np.radians(lat_mean)),
    coords_df['latitude'].values * 111.0
])

# 5. Construction du KDTree et recherche des voisins dans un rayon de 30 km
tree = cKDTree(coords_km)
RADIUS_KM = 30.0
neighbors_list = tree.query_ball_tree(tree, r=RADIUS_KM)

# 6. Propagation spatiale décalée d'un mois (M-1)
contagion_matrix = np.zeros_like(pivoted_fires, dtype=np.float32)

for i in range(n_communes):
    neighbors_excl = [n for n in neighbors_list[i] if n != i]
    if neighbors_excl:
        contagion_matrix[1:, i] = pivoted_fires[:-1, neighbors_excl].sum(axis=1)

# 7. Reconstruction du DataFrame et fusion
contagion_df = pd.DataFrame(
    contagion_matrix,
    index=pivot_df.index,
    columns=commune_codes
).unstack().reset_index()

contagion_df.columns = ['code_insee', 'date_idx', 'indice_contagion_voisinage_last_1m']

df_spatio_temp = pd.merge(df_spatio_temp, contagion_df, on=['code_insee', 'date_idx'], how='left')
df_spatio_temp['indice_contagion_voisinage_last_1m'] = df_spatio_temp['indice_contagion_voisinage_last_1m'].fillna(0).astype('float32')
df_spatio_temp.drop(columns=['date_idx'], inplace=True)

print(" Indice de contagion spatiale à M-1 calculé avec succès.")


In [ ]:
# Variables calendaires connues à l'avance, agrégées au niveau commune-mois.
df_spatio_temp = add_monthly_calendar_features(df_spatio_temp)

calendar_columns = [
    'nb_jours_dans_mois', 'nb_jours_weekend', 'nb_jours_feries',
    'nb_jours_vacances', 'nb_jours_off', 'nb_jours_holy_or_off',
    'ratio_jours_off', 'ratio_jours_vacances'
]
assert df_spatio_temp[calendar_columns].notna().all().all()
print(f"Features calendaires ajoutées : {calendar_columns}")

In [ ]:
# Optimisation des types de données.
type_map = {
    'code_insee': 'category',
    'annee': 'int16',
    'mois': 'int8',
    'nb_incendies': 'int16',
    'target_occurrence': 'int8',
    'is_summer_peak': 'int8',
    'is_spring_peak': 'int8',
    'feux_commune_last_1m': 'int8',
    'feux_commune_last_2m': 'int8',
    'feux_commune_last_3m': 'int8',
    'nb_jours_dans_mois': 'int8',
    'nb_jours_weekend': 'int8',
    'nb_jours_feries': 'int8',
    'nb_jours_vacances': 'int8',
    'nb_jours_off': 'int8',
    'nb_jours_holy_or_off': 'int8',
}

for column, dtype in type_map.items():
    if column in df_spatio_temp.columns:
        df_spatio_temp[column] = df_spatio_temp[column].astype(dtype)

float_columns = df_spatio_temp.select_dtypes(include=['float64']).columns
if len(float_columns):
    df_spatio_temp[float_columns] = df_spatio_temp[float_columns].astype('float32')

print(f"Empreinte mémoire optimisée : {df_spatio_temp.memory_usage(deep=True).sum() / 1e6:.1f} Mo")

Empreinte mémoire optimisée : 171.9 Mo


In [ ]:
df_spatio_temp.shape

(2321280, 25)

In [ ]:
df_spatio_temp.head()

,code_insee,annee,mois,nb_incendies,surface_brulee_ha,target_occurrence,target_log_surface,sin_mois,cos_mois,is_summer_peak,...,densite,altitude_moyenne,feux_commune_last_12m,surface_commune_last_12m,feux_commune_last_1m,feux_commune_last_2m,feux_commune_last_3m,score_risque_commune_global,score_risque_commune_mensuel,indice_contagion_voisinage_last_1m
0,01014,2006,1,0,0.0,0,0.0,0.5,0.866025,0,...,144.0,709,0.0,0.0,0,0,0,0.010417,0.0,0.0
1,01015,2006,1,0,0.0,0,0.0,0.5,0.866025,0,...,51.0,289,0.0,0.0,0,0,0,0.005208,0.0,0.0
2,01017,2006,1,0,0.0,0,0.0,0.5,0.866025,0,...,55.0,517,0.0,0.0,0,0,0,0.005208,0.0,0.0
3,01032,2006,1,0,0.0,0,0.0,0.5,0.866025,0,...,258.0,223,0.0,0.0,0,0,0,0.005208,0.0,0.0
4,01034,2006,1,0,0.0,0,0.0,0.5,0.866025,0,...,409.0,278,0.0,0.0,0,0,0,0.005208,0.0,0.0


In [ ]:

# 1. Comptage et proportions
counts = df_spatio_temp['target_occurrence'].value_counts()
proportions = df_spatio_temp['target_occurrence'].value_counts(normalize=True) * 100

print("=== Distribution de la variable cible (target_occurrence) ===")
print(f"Mois sans incendie (0) : {counts.get(0, 0):,} ({proportions.get(0, 0):.2f}%)".replace(",", " "))
print(f"Mois avec incendie (1) : {counts.get(1, 0):,} ({proportions.get(1, 0):.2f}%)".replace(",", " "))

if 1 in counts:
    ratio = int(counts[0] / counts[1])
    print(f"\nRatio de déséquilibre : 1 mois avec feu pour environ {ratio} mois normaux.")


=== Distribution de la variable cible (target_occurrence) ===
Mois sans incendie (0) : 2 279 185 (98.19%)
Mois avec incendie (1) : 42 095 (1.81%)

Ratio de déséquilibre : 1 mois avec feu pour environ 54 mois normaux.


In [ ]:
if not SPATIO_TEMP_DATA_DIR.exists():
    SPATIO_TEMP_DATA_DIR.mkdir(parents=True, exist_ok=True)

output_path = DATA_DIR / 'data_processed' / 'incendies_features_v2.parquet'
metadata = export_versioned_dataset(
    df_spatio_temp,
    output_path,
    dataset_name='incendies_features',
    dataset_version='v2',
    target='target_occurrence',
    granularity='commune-mois',
    source_files=[
        'data/bdiff_data_clean/incendies.parquet',
        'data/geo_data_clean/communes_metropole_corse.parquet',
    ],
)

# Conservation de l'ancien chemin pour les cellules et usages historiques.
df_spatio_temp.to_parquet(
    SPATIO_TEMP_DATA_DIR / 'df_spatio_temp.parquet',
    index=False,
)

print(f"Dataset versionné : {output_path}")
print(f"SHA-256 : {metadata['sha256']}")
print(f"Dimensions : {metadata['n_rows']} lignes x {metadata['n_columns']} colonnes")

✅ Export réussi : 'df_spatio_temp.parquet' a été créé !
